In [ ]:
%%bash
cd ..
jupyter nbconvert --log-level=ERROR --execute --to notebook --inplace --ExecutePreprocessor.record_timing=False n1-parse-refs.ipynb
jupyter nbconvert --log-level=ERROR --clear-output --inplace n1-parse-refs.ipynb
cd misc

In [ ]:
# %%bash
# exec_annotation -p ../tmp/profiles -k ../tmp/ko_list -o ../tmp/kofam.txt ../tmp/seq.fa --cpu 10 -f detail-tsv
# exec_annotation -p ../tmp/profiles -k ../tmp/ko_list -o ../tmp/kofam2.txt ../tmp/new.fa --cpu 10 -f detail-tsv

In [ ]:
# %%bash
# diamond blastp \
#     -q ../tmp/seq.fa \
#     -d ../tmp/seq.fa \
#     --out ../tmp/hit_seq.txt \
#     --outfmt 6 qseqid sseqid nident qlen slen pident qcovhsp scovhsp bitscore evalue \
#     --subject-cover 95 --query-cover 95 \
#     -k 0 --threads 48 --no-self-hits --masking 0 --quiet

In [ ]:
from Bio import SeqIO
import pandas as pd
import os
import shutil

seq2description = dict()
with open('../tmp/seq2description.fa') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        seq2description[record.seq] = record.description.split(' >')[0]

seq2source = dict()
with open('../tmp/seq2source.fa') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        seq2source[record.seq] = record.id

qset = set()
with open('../tmp/hit_seq.txt') as f:
    for line in f:
        ls = line.rstrip().split('\t')
        if ls[0].split('|')[2] != ls[1].split('|')[2] and round(float(ls[5])) >= 95:
            if '*' in ls[0]:
                qset.add(ls[0])

ko = []
with open('../tmp/kofam.txt') as f:
    for line in f:
        if line[0] != '#':
            ls = line.rstrip().split('\t')
            if ls[3] != '':
                ratio = float(ls[4]) / float(ls[3])
                ko.append((ls[1], ratio, ls[2] + ls[0] + '|' + ls[-1].split('"')[1].split(' [')[0]))
ko = pd.DataFrame(ko, columns = ['source', 'ratio', 'ko']).sort_values('ratio', ascending=False).groupby('source')['ko'].first().to_dict()

records = []
lines = []
with open('../tmp/seq.fa') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        if record.id not in qset:
            desc = seq2description.get(record.seq, 'NA')
            if not desc:
                continue

            line = [record.id, ko.get(record.id), seq2source.get(record.seq), desc.split(' ', 1)[-1].split(' [')[0].split('MULTISPECIES: ')[-1]]
            record.id = '|'.join(record.id.split('|')[:3] + [desc.split(' ', 1)[0]])
            record.description = desc.split(' ', 1)[-1]
            records.append(record)
    
            lines.append(line + [record.id])

with open('../sarg_ref.fa', 'w') as output_handle:
    SeqIO.write(records, output_handle, 'fasta')

summary = pd.DataFrame(lines, columns = ['id', 'ko', 'source', 'description', 'sarg'])
summary['type'] = summary['id'].str.split('|').str.get(1)
summary['subtype'] = summary['id'].str.split('|').str.get(2)
summary['accession'] = summary['id'].str.split('|').str.get(3)
summary['definition'] = summary.ko.str.split('|').str.get(1)
summary['ko'] = summary.ko.str.split('|').str.get(0)

cols = ['type', 'subtype', 'sarg', 'source', 'description', 'ko', 'definition']
summary[cols].sort_values(cols).to_csv('summary.tsv', index=False, sep='\t')

In [ ]:
%%bash
mmseqs easy-cluster ../sarg_ref.fa ../tmp/sarg_ref ../tmp/TMP \
    --cov-mode 0 -c 0.95 --min-seq-id 0.95 --cluster-reassign \
    --threads 32 > /dev/null

In [ ]:
c = pd.read_table('../tmp/sarg_ref_cluster.tsv', header=None)
c['left'] = c[0].str.split('|').str.get(-2)
c['right'] = c[1].str.split('|').str.get(-2)

c['id'] = c.left == c.right
c[0] = c[0].str.split('|').str.get(-1)
c[1] = c[1].str.split('|').str.get(-1)

cc = c[c.id==False].groupby(['left',  'right'], as_index=False).size()
if len(cc) > 0:
    print(cc)

In [ ]:
if not os.path.isdir('../info'):
    os.makedirs('../info')
else:
    shutil.rmtree('../info')
    os.makedirs('../info')

viewed = []
try:
    with open('../tmp/viewed.txt') as f:
        for line in f:
            viewed.extend(line.rstrip().split(' '))
except:
    pass

summary = pd.read_table('../misc/summary.tsv')
cols = ['type', 'subtype', 'source', 'sarg', 'description', 'ko']
for row, group in summary[~(summary.type.isin(viewed) | (summary.subtype.isin(viewed)))].groupby('type'):
    group['misc'] = '"' + group['source'] + '", // ' + group.description
    group.sort_values(cols).set_index(['type', 'subtype', 'source']).to_excel(f"../info/{row[:31].replace('/', '-')}.xlsx")

In [ ]:
ss = summary.groupby('subtype').type.nunique()
if len(ss[ss > 1 ]) > 0:
    print(ss[ ss > 1])

In [ ]:
# df = pd.read_table('tmp/hit_seq.txt', header=None)
# df = df[df[0].str.contains('tetracycline')]
# df = df[df[0].str.contains('/', regex=False)]
# df = df[~df[1].str.contains('/', regex=False)]
# df = df.groupby(0).first()

# sarg2source = pd.read_table('misc/summary.tsv').set_index('sarg').source.to_dict()
# for i,j in df.iterrows():
#     print(f'"{sarg2source.get(i)}", // {sarg2source.get(j[1])} | id: {j[5]} qcov: {j[6]} scov: {j[7]}')